# TartuNLP (entrenado en ruso) sobre el dev de SemEval-2027 – Subtask 2

Pipeline: **convertir** datos → **predecir** con el adaptador → **evaluar** F1.

Requisitos en este runtime de Colab:
- el entorno `/content/venv` del notebook de reproducción (si el runtime es nuevo, vuelve a ejecutar sus celdas de instalación),
- el adaptador entrenado (en Drive si usaste esa celda, o en `/content/ancient-lang-adapters/trained_models/ru`).

## 0. Traer nuestro repo al runtime
El kernel de Colab es una máquina remota: **no ve los archivos de tu ordenador**, aunque uses VS Code. Por eso clonamos el repo (haz `git push` antes).

In [ ]:
REPO_URL = 'https://github.com/USUARIO/REPO.git'  # <-- cambiar (privado: https://<TOKEN>@github.com/...)
REPO_DIR = '/content/repo'
import os
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

## 1. Drive: datos dev y adaptador
Sube la carpeta `dev/` (con `SV/` dentro) a `MyDrive/semeval27/dev`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/semeval27/dev'
ADAPTER = '/content/drive/MyDrive/axolotl_models/ru/classification'
# Si no guardaste el modelo en Drive:
# ADAPTER = '/content/ancient-lang-adapters/trained_models/ru/classification'
!ls {DATA_DIR} && ls {ADAPTER}

## 2. Convertir a pares (definición, uso)

In [ ]:
!python scripts/convert_data.py --data_dir {DATA_DIR} --out_dir outputs/pairs

## 3. Predecir con el adaptador de TartuNLP
Usa el Python del entorno de TartuNLP (`/content/venv`).

In [ ]:
!/content/venv/bin/python scripts/predict_tartunlp.py \
    --pairs outputs/pairs/SV.pairs.tsv \
    --adapter_dir {ADAPTER} \
    --out outputs/preds/SV.tartunlp_ru.tsv

## 4. Evaluar
Umbral 0.35 (el de TartuNLP) + barrido de umbrales. El mejor umbral del barrido está elegido sobre el propio dev: es optimista.

In [ ]:
!python scripts/evaluate_subtask2.py --pred outputs/preds/SV.tartunlp_ru.tsv --threshold 0.35 --sweep

## 5. Guardar resultados en el repo (opcional)

In [ ]:
!git status --short results/
# !git add results/ && git commit -m 'Resultados TartuNLP-ru en dev SV' && git push